# 컴퓨터비전 과제 #2

## 과제#1 이후부터 현재까지 실습 코드를 모두 수행하고 결과 출력 (2점)

- 각 수행마다 코드에는 어떤 과정인지 주석 처리

**실습 순서**
1. SIFT 실습
2. ORB 실습
3. Similarity Search 실습
4. RANSAC 실습
5. Blocking Matching 실습
6. 2.5D 영상 실습

## [실습 1] SIFT 실습

Butterfly 영상을 강제로 회전·축소시킨 뒤, SIFT 특징점을 검출하고 FLANN(KDTree) 매처로 매칭한다. Lowe's ratio test로 좋은 매칭만 선별하여 시각화한다.

In [ ]:
import cv2
import numpy as np
import urllib.request
from google.colab.patches import cv2_imshow

# 이미지 다운로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/butterfly.jpg'
urllib.request.urlretrieve(url, 'butterfly.jpg')
img = cv2.imread('butterfly.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 비교를 위해 이미지를 강제로 회전 및 축소시킴
rows, cols = gray.shape
M = cv2.getRotationMatrix2D((cols/2, rows/2), 45, 0.7) # 45도 회전, 0.7배 축소
img_varied = cv2.warpAffine(gray, M, (cols, rows))

# SIFT 객체 생성 및 특징점(Keypoints) 검출
sift = cv2.SIFT_create()
# kp: 특징점 위치/크기 정보, des: 특징점을 설명하는 128차원 벡터(Descriptor)
kp1, des1 = sift.detectAndCompute(gray, None)
kp2, des2 = sift.detectAndCompute(img_varied, None)

# 특징점 매칭 (FLANN Matcher 사용)
# trees=5: 5개의 무작위 KD-트리를 병렬로 만들어 탐색 속도를 높임
# checks=50: 트리를 얼마나 깊게 뒤져볼지 결정
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)

flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1, des2, k=2)   # 가장 가까운 2개 특징 찾음

# 좋은 매칭점만 선별 (Lowe's ratio test)
good_matches = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good_matches.append(m)

# 결과 시각화
# 두 이미지 사이의 매칭 라인을 그림
img_match = cv2.drawMatches(gray, kp1, img_varied, kp2, good_matches, None,
               flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

print(f"검출된 특징점 개수: {len(kp1)}")
print(f"매칭 성공 개수: {len(good_matches)}")
cv2_imshow(img_match)

## [실습 2] ORB 실습

vtest.avi 영상의 첫 프레임에서 보행자가 있는 영역을 타겟 ROI로 설정한 뒤, ORB 특징점과 BF(Hamming) 매처를 이용해 매 프레임에서 추적하며 검출/매칭/총 처리시간 및 FPS를 출력한다.

In [ ]:
import cv2
import numpy as np
import time
import urllib.request
from google.colab.patches import cv2_imshow

# 샘플 영상 로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi'
urllib.request.urlretrieve(url, 'vtest.avi')
cap = cv2.VideoCapture('vtest.avi')

# 타겟 설정 (첫 프레임의 특정 영역)
ret, first_frame = cap.read()
if not ret:
    exit()

target_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
target_roi = target_gray[100:300, 200:400] # 보행자가 있는 영역 절삭

# ORB 및 매처 초기화
orb = cv2.ORB_create(nfeatures=1000)
kp1, des1 = orb.detectAndCompute(target_roi, None)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

print(f"{'Frame':<8} | {'Detect(ms)':<12} | {'Match(ms)':<12} | {'Total(ms)':<12} | {'FPS':<6}")
print("-" * 65)

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    # --- 속도 측정 시작 ---
    start_total = time.time()
    # 특징점 검출 및 기술자 생성 시간 측정
    start_det = time.time()
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    kp2, des2 = orb.detectAndCompute(gray_frame, None)
    end_det = time.time()
    det_time = (end_det - start_det) * 1000 # ms 단위

    match_time = 0
    if des2 is not None:
        # 매칭 시간 측정
        start_match = time.time()
        matches = bf.knnMatch(des1, des2, k=2)
        good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
        end_match = time.time()
        match_time = (end_match - start_match) * 1000

    # --- 속도 측정 종료 ---
    end_total = time.time()
    total_time = (end_total - start_total) * 1000
    fps = 1.0 / (end_total - start_total) if total_time > 0 else 0

    # 지표 출력 (매 10프레임마다)
    if frame_count % 10 == 0:
        print(f"{frame_count:<8} | {det_time:<12.2f} | {match_time:<12.2f} | {total_time:<12.2f} | {fps:<6.1f}")

    # 시각화 (매 100프레임마다)
    if frame_count % 100 == 0:
        res = cv2.drawMatches(target_roi, kp1, frame, kp2, good_matches[:20], None,
                       flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        cv2.putText(res, f"FPS: {fps:.1f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2_imshow(res)

    frame_count += 1

cap.release()

## [실습 3] Similarity Search 실습

Aloe 스테레오 페어에서 SIFT 특징점 10,000개를 추출한 뒤, OpenCV BF/FLANN(CPU)과 Faiss FlatL2/IVF-PQ(GPU) 4가지 매처의 매칭 속도를 벤치마크한다.

**[코랩 GPU 설정 필요]** [수정] → [노트북 설정] → [하드웨어 가속기] → T4 GPU

In [ ]:
# 1. 설치
!pip install faiss-gpu-cu12

In [ ]:
# 2. 설치 확인
import faiss
print(faiss.__version__)
print(faiss.get_num_gpus())

In [ ]:
# 3. 코드실행
import cv2
import numpy as np
import faiss
import time
import urllib.request
from google.colab.patches import cv2_imshow

# GPU 리소스 초기화
res = faiss.StandardGpuResources()

# 이미지 로드 및 특징점 추출 (데이터(특징점)의 수: 10,000개)
url1 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeL.jpg'
url2 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeR.jpg'
urllib.request.urlretrieve(url1, 'aloeL.jpg')
urllib.request.urlretrieve(url2, 'aloeR.jpg')

img1 = cv2.imread('aloeL.jpg', cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread('aloeR.jpg', cv2.IMREAD_GRAYSCALE)

sift = cv2.SIFT_create(nfeatures=10000)
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

des1_f32 = des1.astype('float32')
des2_f32 = des2.astype('float32')

print(f"특징점 개수: Image1({len(kp1)}), Image2({len(kp2)})\n")

def run_benchmark_and_show(name, search_func, is_faiss=False):
    # --- 순수 속도 측정 (시각화 제외) ---
    search_func()  # 워밍업 (GPU 메모리 할당 및 초기화 방지)
    start_time = time.time()
    for _ in range(10):
        raw_result = search_func()
    end_time = time.time()
    avg_ms = ((end_time - start_time) / 10) * 1000
    print(f"[{name:<18}] 순수 매칭 속도: {avg_ms:>7.2f} ms")

    # --- [B] 시각화를 위한 후처리 (측정 범위 밖) ---
    good_matches = []
    if not is_faiss:  # OpenCV Matcher (BF, FLANN)
        for m, n in raw_result:
            if m.distance < 0.7 * n.distance:
                good_matches.append(m)
    else:  # Faiss (Flat, PQ)
        D, I = raw_result
        for i in range(len(des1_f32)):
            if D[i][0] < 0.7 * D[i][1]:
                m = cv2.DMatch(_queryIdx=i, _trainIdx=int(I[i][0]), _distance=D[i][0])
                good_matches.append(m)

    # 매칭 결과 그리기
    res_img = cv2.drawMatches(img1, kp1, img2, kp2, good_matches[:1000], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    cv2_imshow(res_img)
    print(f"추출된 유효 매칭 수: {len(good_matches)}\n" + "-"*80)

# OpenCV Brute-Force (CPU)
bf = cv2.BFMatcher(cv2.NORM_L2)
run_benchmark_and_show("BF-Matcher (CPU)", lambda: bf.knnMatch(des1, des2, k=2))

# OpenCV FLANN (CPU)
flann = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))
run_benchmark_and_show("FLANN (CPU)", lambda: flann.knnMatch(des1, des2, k=2))

# Faiss FlatL2 (GPU)
index_flat = faiss.IndexFlatL2(128)
gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, index_flat)
gpu_index_flat.add(des2_f32)
run_benchmark_and_show("Faiss Flat (GPU)", lambda: gpu_index_flat.search(des1_f32, 2), is_faiss=True)

# Faiss IVF-PQ (GPU)
# 128차원을 16개 조각(m=16)으로 나누어 8비트(nbits=8)로 양자화
nlist = 100
m = 16
quantizer = faiss.IndexFlatL2(128)
index_pq = faiss.IndexIVFPQ(quantizer, 128, nlist, m, 8)
gpu_index_pq = faiss.index_cpu_to_gpu(res, 0, index_pq)
gpu_index_pq.train(des2_f32)
gpu_index_pq.add(des2_f32)
run_benchmark_and_show("Faiss IVF-PQ (GPU)", lambda: gpu_index_pq.search(des1_f32, 2), is_faiss=True)

## [실습 4] RANSAC 실습

Aloe 스테레오 페어에서 SIFT 특징점 5,000개를 추출하고 Faiss-GPU로 KNN 매칭한 뒤 Lowe's ratio test로 1차 필터링한다. 이후 RANSAC으로 호모그래피 행렬을 추정해 인라이어(올바른 매칭)만 남긴다.

In [ ]:
import cv2
import numpy as np
import faiss
import urllib.request
from google.colab.patches import cv2_imshow

# 1. GPU 리소스 초기화 및 이미지 로드
res = faiss.StandardGpuResources()

# 서로 다른 시점에서 촬영된 이미지 두 장
url1 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeL.jpg'
url2 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeR.jpg'
urllib.request.urlretrieve(url1, 'aloeL.jpg')
urllib.request.urlretrieve(url2, 'aloeR.jpg')

img1_gray = cv2.imread('aloeL.jpg', cv2.IMREAD_GRAYSCALE)
img2_gray = cv2.imread('aloeR.jpg', cv2.IMREAD_GRAYSCALE)

# 2. SIFT 특징점 추출 (반복 패턴이라 특징점 개수를 충분히 늘림)
sift = cv2.SIFT_create(nfeatures=5000)
kp1, des1 = sift.detectAndCompute(img1_gray, None)
kp2, des2 = sift.detectAndCompute(img2_gray, None)

des1_f32 = des1.astype('float32')
des2_f32 = des2.astype('float32')

print(f"특징점 개수: Image1({len(kp1)}개), Image2({len(kp2)}개)\n")

# 3. Faiss-GPU를 이용한 고속 KNN 매칭 (K=2)
index = faiss.IndexFlatL2(128)
gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
gpu_index.add(des2_f32)
distances, indices = gpu_index.search(des1_f32, 2)

# 4. Lowe's Ratio Test를 통한 1차 필터링
good_matches = []
for i in range(len(des1)):
    if distances[i][0] < 0.5 * distances[i][1]:
        m = cv2.DMatch(_queryIdx=i, _trainIdx=int(indices[i][0]),
                       _distance=distances[i][0])
        good_matches.append(m)

print(f"Faiss 매칭 완료 | Ratio Test 통과: {len(good_matches)}개")

# 5. RANSAC + Homography 계산 및 최종 시각화
if len(good_matches) > 4:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # RANSAC으로 homography 행렬(M) 계산
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 3.0)
    matches_mask = mask.ravel().tolist()

    # Inliers만 추출
    inlier_matches = [good_matches[i] for i in range(len(matches_mask)) if matches_mask[i] == 1]
    print(f"RANSAC 검증 완료 | 최종 인라이어(Inliers): {len(inlier_matches)}개")

    res_img = cv2.drawMatches(img1_gray, kp1, img2_gray, kp2, inlier_matches[:500], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    print("\n[RANSAC 매칭 결과]")
    cv2_imshow(res_img)
else:
    print("매칭점이 부족하여 호모그래피를 계산할 수 없습니다.")

## [실습 5] Blocking Matching 실습

Aloe 스테레오 페어에 OpenCV `cv2.StereoBM`(SAD 기반)을 적용하여 양안시차(disparity) 지도를 계산하고, jet colormap으로 시각화한다.

In [ ]:
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
import urllib.request

# Aloe 이미지 다운로드
url1 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeL.jpg'
url2 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeR.jpg'
urllib.request.urlretrieve(url1, 'aloeL.jpg')
urllib.request.urlretrieve(url2, 'aloeR.jpg')

imgL_color = cv2.imread('aloeL.jpg')
imgL_gray = cv2.imread('aloeL.jpg', cv2.IMREAD_GRAYSCALE)
imgR_gray = cv2.imread('aloeR.jpg', cv2.IMREAD_GRAYSCALE)

# 원본 해상도 저장
orig_h, orig_w = imgL_gray.shape

# StereoBM 설정 (SAD 기반)
num_disp = 128
block_size = 15
stereo = cv2.StereoBM_create(numDisparities=num_disp, blockSize=block_size)

# disparity 계산
start_time = time.time()
disparity = stereo.compute(imgL_gray, imgR_gray)
elapsed_time = time.time() - start_time

# StereoBM 결과는 16배 스케일이므로 실제 픽셀 단위로 변환
disparity_float = disparity.astype(np.float32) / 16.0

# 시각화를 위한 정규화 (0~255)
# 시차 값이 0보다 작은 영역(무효값)은 제외
disparity_vis = cv2.normalize(disparity, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

# 결과 출력
print(f"이미지 크기: {orig_w}x{orig_h}")
print(f"소요 시간: {elapsed_time:.4f}초")

plt.figure(figsize=(20, 10))

# 왼쪽 원본 이미지
plt.subplot(1, 2, 1)
plt.title("Original Left Image")
plt.imshow(cv2.cvtColor(imgL_color, cv2.COLOR_BGR2RGB))
plt.axis('off')

# 원본 크기의 Disparity Map
plt.subplot(1, 2, 2)
plt.title(f"Full-Res SAD Disparity Map\n(NumDisp: {num_disp}, Block: {block_size})")
plt.imshow(disparity_vis, cmap='jet')
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis('off')

plt.tight_layout()
plt.show()

## [실습 6] 2.5D 영상 실습

실습 5에서 계산한 disparity_float을 이용하여 각 픽셀의 3D 좌표(X, Y, Z)를 역투영하고, 컬러 정보와 결합한 포인트 클라우드를 PLY 포맷으로 저장한다. (Meshlab 등에서 열어 3D 시각화 가능)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. 이전 단계에서 생성한 데이터 준비 (imgL_color, disparity_float 사용)
scale = 1.0
img_small = cv2.resize(imgL_color, None, fx=scale, fy=scale)
disp_small = cv2.resize(disparity_float, None, fx=scale, fy=scale)

h, w = disp_small.shape

# 초점거리(f)와 베이스라인(B)은 임의의 값을 설정
f = 0.8 * w  # 가상의 초점 거리
B = 1.0      # 가상의 카메라 간격

# 3. 3D 좌표 계산
x_coords, y_coords = np.meshgrid(np.arange(w), np.arange(h))

mask = disp_small > 0.5

# Z = (f * B) / disparity
z = np.zeros_like(disp_small)
z[mask] = (f * B) / disp_small[mask]

# X, Y 좌표 역투영
x = (x_coords - w/2) * z / f
y = (y_coords - h/2) * z / f

# 4. 시각화 데이터 준비 (유효한 점만 필터링)
points_x = x[mask].ravel()
points_y = y[mask].ravel()
points_z = z[mask].ravel()

# 색상 정보 추출 (BGR -> RGB 및 0~1 정규화)
colors = cv2.cvtColor(img_small, cv2.COLOR_BGR2RGB)[mask].reshape(-1, 3) / 255.0

def save_point_cloud_ply(filename, points_x, points_y, points_z, colors):
    """
    포인트 클라우드 데이터를 PLY 파일로 저장
    points_x, y, z: 1차원 넘파이 배열 (좌표)
    colors: 1차원 넘파이 배열 (0~1 범위의 RGB 값)
    """
    # 색상을 0~255 범위의 정수로 변환
    colors_int = (colors * 255).astype(np.uint8)

    # 헤더 작성
    num_points = len(points_x)
    header = f"""ply
format ascii 1.0
element vertex {num_points}
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
end_header
"""
    # 데이터 결합 및 저장
    with open(filename, 'w') as f:
        f.write(header)
        for i in range(num_points):
            # 좌표와 색상을 한 줄씩 기록
            f.write(f"{points_x[i]} {points_y[i]} {points_z[i]} {colors_int[i, 0]} {colors_int[i, 1]} {colors_int[i, 2]}\n")

    print(f"성공: '{filename}' 파일이 저장되었습니다. (총 {num_points}개의 점)")

# 파일 저장 실행
save_point_cloud_ply("aloe_2_5d.ply", points_x, points_y, points_z, colors)